## BB-code circuit-level noise Monte Carlo (threaded version)

Reproduces Figure 3a from arXiv:2308.07915 for the [[72,12,6]] BB code.

**Important**: Launch Julia with threads enabled: `JULIA_NUM_THREADS=96 jupyter notebook`
or set `-t 96` in the kernel spec. Verify with `Threads.nthreads()` below.

In [1]:
using Plots
using LinearAlgebra
using SparseArrays
using Random
using JLD2
using LatticeAlgorithms
using Base.Threads
println("nthreads = $(nthreads())")

nthreads = 96


## Parameters

In [2]:
num_samples = Int(5e3)
num_threads = nthreads()
num_samples_per_thread = Int(ceil(num_samples / num_threads))
num_total_samples = num_samples_per_thread * num_threads

osd_order = 1

p_range = [1e-3, 2e-3, 3e-3, 4e-3, 5e-3, 6e-3]

l, m = 6, 6
A_terms = [(:x, 3), (:y, 1), (:y, 2)]
B_terms = [(:y, 3), (:x, 1), (:x, 2)]
d = 6
num_cycles = d
k = 12;

## Simulation helper functions

In [3]:
function sim_Z(C, lin, TOTAL)
    syn = Int[]; smap = Dict{Tuple{Symbol,Int},Vector{Int}}()
    state = zeros(Int, TOTAL); sc = 0
    for g in C
        op = g[1]
        if     op==:CNOT;  state[lin[g[2]]] ⊻= state[lin[g[3]]]
        elseif op==:PrepX; state[lin[g[2]]] = 0
        elseif op==:MeasX; sc+=1; push!(syn, state[lin[g[2]]]); push!(get!(Vector{Int},smap,g[2]),sc)
        elseif op in (:Z,:Y);           state[lin[g[2]]] ⊻= 1
        elseif op in (:ZX,:YX);         state[lin[g[2]]] ⊻= 1
        elseif op in (:XZ,:XY);         state[lin[g[3]]] ⊻= 1
        elseif op in (:ZZ,:YY,:YZ,:ZY); state[lin[g[2]]] ⊻= 1; state[lin[g[3]]] ⊻= 1
        end
    end
    syn, state, smap
end

function sim_X(C, lin, TOTAL)
    syn = Int[]; smap = Dict{Tuple{Symbol,Int},Vector{Int}}()
    state = zeros(Int, TOTAL); sc = 0
    for g in C
        op = g[1]
        if     op==:CNOT;  state[lin[g[3]]] ⊻= state[lin[g[2]]]
        elseif op==:PrepZ; state[lin[g[2]]] = 0
        elseif op==:MeasZ; sc+=1; push!(syn, state[lin[g[2]]]); push!(get!(Vector{Int},smap,g[2]),sc)
        elseif op in (:X,:Y);           state[lin[g[2]]] ⊻= 1
        elseif op in (:XZ,:YZ);         state[lin[g[2]]] ⊻= 1
        elseif op in (:ZX,:ZY);         state[lin[g[3]]] ⊻= 1
        elseif op in (:XX,:YY,:XY,:YX); state[lin[g[2]]] ⊻= 1; state[lin[g[3]]] ⊻= 1
        end
    end
    syn, state, smap
end

function sparsify_syn!(syn, smap, checks, nr)
    for c in checks
        pos = smap[c]; prev = syn[pos[1]]
        for row in 2:nr
            cur = syn[pos[row]]; syn[pos[row]] = (cur+prev)&1; prev = cur
        end
    end
end

function generate_noisy_circuit(cycle_rep, p, rng)
    circ = Tuple[]
    for gate in cycle_rep
        op = gate[1]
        if op == :MeasX
            rand(rng) <= p && push!(circ, (:Z, gate[2]))
            push!(circ, gate)
        elseif op == :MeasZ
            rand(rng) <= p && push!(circ, (:X, gate[2]))
            push!(circ, gate)
        elseif op == :PrepX
            push!(circ, gate)
            rand(rng) <= p && push!(circ, (:Z, gate[2]))
        elseif op == :PrepZ
            push!(circ, gate)
            rand(rng) <= p && push!(circ, (:X, gate[2]))
        elseif op == :IDLE
            if rand(rng) <= p
                push!(circ, ((:X,:Y,:Z)[rand(rng,1:3)], gate[2]))
            end
        elseif op == :CNOT
            push!(circ, gate)
            if rand(rng) <= p
                paulis = [
                    (:X,gate[2]),(:Y,gate[2]),(:Z,gate[2]),
                    (:X,gate[3]),(:Y,gate[3]),(:Z,gate[3]),
                    (:XX,gate[2],gate[3]),(:YY,gate[2],gate[3]),(:ZZ,gate[2],gate[3]),
                    (:XY,gate[2],gate[3]),(:YX,gate[2],gate[3]),
                    (:YZ,gate[2],gate[3]),(:ZY,gate[2],gate[3]),
                    (:XZ,gate[2],gate[3]),(:ZX,gate[2],gate[3]),
                ]
                push!(circ, paulis[rand(rng,1:15)])
            end
        end
    end
    circ
end

function run_trial(p, s, rng)
    nc = s.num_cycles
    cycle_rep = reduce(vcat, fill(s.cycle, nc))
    ntail = vcat(s.cycle, s.cycle)
    nr = nc + 2

    circ = generate_noisy_circuit(cycle_rep, p, rng)
    full = vcat(circ, ntail)

    # Z-sector
    syn_z, st_z, sm_z = sim_Z(full, s.lin, s.TOTAL)
    lact_z = mod.(s.lx * [st_z[s.lin[q]] for q in s.dqubits], 2)
    sparsify_syn!(syn_z, sm_z, s.Xchks, nr)
    rz = bp_osd_cs_decode(s.HdecZ_dense, syn_z, s.cpZ;
        osd_order=s.osd_order, λ=60, check_to_bit_update_rule=:min_sum, min_sum_scaling=:roffe)
    lgz = mod.(s.HZ_full * rz.error, 2)[s.flrZ+1 : s.flrZ+s.k]
    lgz != lact_z && return false

    # X-sector
    syn_x, st_x, sm_x = sim_X(full, s.lin, s.TOTAL)
    lact_x = mod.(s.lz * [st_x[s.lin[q]] for q in s.dqubits], 2)
    sparsify_syn!(syn_x, sm_x, s.Zchks, nr)
    rx = bp_osd_cs_decode(s.HdecX_dense, syn_x, s.cpX;
        osd_order=s.osd_order, λ=60, check_to_bit_update_rule=:min_sum, min_sum_scaling=:roffe)
    lgx = mod.(s.HX_full * rx.error, 2)[s.flrX+1 : s.flrX+s.k]
    return lgx == lact_x
end

run_trial (generic function with 1 method)

## Decoder setup

In [4]:
function decoder_setup(l, m, A_terms, B_terms, num_cycles, error_rate; osd_order=0)
    n = bb_num_data_qubits(l, m)
    n2 = l * m; k = 0; TOTAL = 2n

    HX, HZ = bb_check_matrices(l, m, A_terms, B_terms)
    lx, lz = css_logicals(HX, HZ)
    k = size(lx, 1)

    x_mat = bb_x_matrix(l, m); y_mat = bb_y_matrix(l, m)
    pow(kind, power) = kind==:x ? Int64.(x_mat^mod(power,l)) :
                       kind==:y ? Int64.(y_mat^mod(power,m)) :
                       Matrix{Int64}(I, n2, n2)
    A1,A2,A3 = pow(A_terms[1]...), pow(A_terms[2]...), pow(A_terms[3]...)
    B1,B2,B3 = pow(B_terms[1]...), pow(B_terms[2]...), pow(B_terms[3]...)

    lin = Dict{Tuple{Symbol,Int},Int}()
    Xchks = Tuple{Symbol,Int}[]; dqubits = Tuple{Symbol,Int}[]; Zchks = Tuple{Symbol,Int}[]
    cnt = 1
    for i in 0:n2-1; q=(:Xcheck,i);    push!(Xchks,q);   lin[q]=cnt; cnt+=1; end
    for i in 0:n2-1; q=(:data_left,i);  push!(dqubits,q); lin[q]=cnt; cnt+=1; end
    for i in 0:n2-1; q=(:data_right,i); push!(dqubits,q); lin[q]=cnt; cnt+=1; end
    for i in 0:n2-1; q=(:Zcheck,i);     push!(Zchks,q);   lin[q]=cnt; cnt+=1; end

    nbs = Dict{Tuple{Tuple{Symbol,Int},Int}, Tuple{Symbol,Int}}()
    for i in 0:n2-1
        c = (:Xcheck, i)
        nbs[(c,0)]=(:data_left,  findfirst(==(1),A1[i+1,:])-1)
        nbs[(c,1)]=(:data_left,  findfirst(==(1),A2[i+1,:])-1)
        nbs[(c,2)]=(:data_left,  findfirst(==(1),A3[i+1,:])-1)
        nbs[(c,3)]=(:data_right, findfirst(==(1),B1[i+1,:])-1)
        nbs[(c,4)]=(:data_right, findfirst(==(1),B2[i+1,:])-1)
        nbs[(c,5)]=(:data_right, findfirst(==(1),B3[i+1,:])-1)
    end
    for i in 0:n2-1
        c = (:Zcheck, i)
        nbs[(c,0)]=(:data_left,  findfirst(==(1),B1[:,i+1])-1)
        nbs[(c,1)]=(:data_left,  findfirst(==(1),B2[:,i+1])-1)
        nbs[(c,2)]=(:data_left,  findfirst(==(1),B3[:,i+1])-1)
        nbs[(c,3)]=(:data_right, findfirst(==(1),A1[:,i+1])-1)
        nbs[(c,4)]=(:data_right, findfirst(==(1),A2[:,i+1])-1)
        nbs[(c,5)]=(:data_right, findfirst(==(1),A3[:,i+1])-1)
    end

    sX = [nothing, 1, 4, 3, 5, 0, 2]; sZ = [3, 5, 0, 1, 2, 4, nothing]
    cycle = Tuple[]
    for q in Xchks; push!(cycle, (:PrepX, q)); end
    cnoted = Set{Tuple{Symbol,Int}}()
    for tgt in Zchks; ctrl=nbs[(tgt,sZ[1])]; push!(cycle,(:CNOT,ctrl,tgt)); push!(cnoted,ctrl); end
    for q in dqubits; q ∉ cnoted && push!(cycle, (:IDLE, q)); end
    for t in 2:6
        for ctrl in Xchks; push!(cycle, (:CNOT, ctrl, nbs[(ctrl,sX[t])])); end
        for tgt in Zchks;  push!(cycle, (:CNOT, nbs[(tgt,sZ[t])], tgt)); end
    end
    for q in Zchks; push!(cycle, (:MeasZ, q)); end
    cnoted2 = Set{Tuple{Symbol,Int}}()
    for ctrl in Xchks; tgt=nbs[(ctrl,sX[7])]; push!(cycle,(:CNOT,ctrl,tgt)); push!(cnoted2,tgt); end
    for q in dqubits; q ∉ cnoted2 && push!(cycle, (:IDLE, q)); end
    for q in dqubits; push!(cycle, (:IDLE, q)); end
    for q in Xchks;   push!(cycle, (:MeasX, q)); end
    for q in Zchks;   push!(cycle, (:PrepZ, q)); end

    println("  Gates/cycle: $(length(cycle))")

    cycle_rep = reduce(vcat, fill(cycle, num_cycles))
    ntail = vcat(cycle, cycle)
    nr = num_cycles + 2

    function enumerate_sector(sim_fn, checks, logical_mat, make_faults)
        Hdict = Dict{Vector{Int},Vector{Int}}(); probs = Float64[]; fidx = 0
        for t in 1:length(cycle_rep)
            gate = cycle_rep[t]
            before = cycle_rep[1:t-1]
            after = cycle_rep[t+1:end]
            op = gate[1]
            for (prob, err_ops) in make_faults(gate, error_rate)
                fidx += 1
                full_faulty =
                    if op in (:MeasX, :MeasZ)
                        vcat(before, err_ops, [gate], after, ntail)
                    elseif op in (:PrepX, :PrepZ, :CNOT)
                        vcat(before, [gate], err_ops, after, ntail)
                    elseif op == :IDLE
                        vcat(before, err_ops, after, ntail)
                    else
                        error("Unsupported gate type: $op")
                    end
                syn, state, smap = sim_fn(full_faulty, lin, TOTAL)
                dstate = [state[lin[q]] for q in dqubits]
                lsyn = mod.(logical_mat * dstate, 2)
                sparsify_syn!(syn, smap, checks, nr)
                supp = findall(!=(0), vcat(syn, lsyn))
                haskey(Hdict,supp) ? push!(Hdict[supp],fidx) : (Hdict[supp]=[fidx])
                push!(probs, prob)
            end
        end
        nsyn = n2*nr; ncols = length(Hdict)
        Hr=Int[]; Hc=Int[]; Hv=Int[]; Fr=Int[]; Fc=Int[]; Fv=Int[]; cp=Float64[]; col=0
        for (supp, fi) in Hdict
            col += 1
            for idx in supp
                idx<=nsyn && (push!(Hr,idx);push!(Hc,col);push!(Hv,1))
                push!(Fr,idx);push!(Fc,col);push!(Fv,1)
            end
            push!(cp, sum(probs[i] for i in fi))
        end
        sparse(Hr,Hc,Hv,nsyn,ncols), sparse(Fr,Fc,Fv,nsyn+k,ncols), cp, nsyn
    end

    z_faults(g,p) = let op=g[1]
        op==:MeasX ? [(p,[(:Z,g[2])])] :
        op==:PrepX ? [(p,[(:Z,g[2])])] :
        op==:IDLE  ? [(p*2/3,[(:Z,g[2])])] :
        op==:CNOT  ? [(p*4/15,[(:Z,g[2])]),(p*4/15,[(:Z,g[3])]),(p*4/15,[(:ZZ,g[2],g[3])])] :
        Tuple[]
    end
    x_faults(g,p) = let op=g[1]
        op==:MeasZ ? [(p,[(:X,g[2])])] :
        op==:PrepZ ? [(p,[(:X,g[2])])] :
        op==:IDLE  ? [(p*2/3,[(:X,g[2])])] :
        op==:CNOT  ? [(p*4/15,[(:X,g[2])]),(p*4/15,[(:X,g[3])]),(p*4/15,[(:XX,g[2],g[3])])] :
        Tuple[]
    end

    println("  Enumerating Z faults...")
    HdecZ, HZ_full, cpZ, flrZ = enumerate_sector(sim_Z, Xchks, lx, z_faults)
    println("    HdecZ: $(size(HdecZ))")
    println("  Enumerating X faults...")
    HdecX, HX_full, cpX, flrX = enumerate_sector(sim_X, Zchks, lz, x_faults)
    println("    HdecX: $(size(HdecX))")

    function filter_nonzero_cols(Hdec, Hfull, cp)
        nz = [j for j in 1:size(Hdec,2) if any(!=(0), Hdec[:,j])]
        Hdec[:, nz], Hfull[:, nz], cp[nz]
    end
    HdecZ, HZ_full, cpZ = filter_nonzero_cols(HdecZ, HZ_full, cpZ)
    HdecX, HX_full, cpX = filter_nonzero_cols(HdecX, HX_full, cpX)
    println("    After filtering: HdecZ $(size(HdecZ)), HdecX $(size(HdecX))")

    rankZ = gf2_rank(Matrix(HdecZ))
    rankX = gf2_rank(Matrix(HdecX))

    cpZ = clamp.(cpZ, 1e-15, 0.5-1e-15)
    cpX = clamp.(cpX, 1e-15, 0.5-1e-15)

    return (
        cycle=cycle, num_cycles=num_cycles, osd_order=osd_order, k=k, TOTAL=TOTAL,
        lin=lin, dqubits=dqubits, Xchks=Xchks, Zchks=Zchks, lx=lx, lz=lz,
        HdecZ_dense=Matrix(HdecZ), HZ_full=Matrix(HZ_full), cpZ=cpZ, flrZ=flrZ,
        HdecX_dense=Matrix(HdecX), HX_full=Matrix(HX_full), cpX=cpX, flrX=flrX,
    )
end

decoder_setup (generic function with 1 method)

## Run

In [ ]:
logfile = "bb_$(2*l*m)_$(k)_$(d)_circuit_level_progress.log"
logf = open(logfile, "w")

function logprintln(logf, msg)
    println(msg)
    println(logf, msg)
    flush(logf)
end

logprintln(logf, "[[$(2*l*m),$(k),$(d)]] circuit-level, num_cycles=$num_cycles, osd_order=$osd_order, samples=$num_total_samples, threads=$(nthreads())")

# One RNG per thread for thread safety
rngs = [MersenneTwister(i) for i in 1:nthreads()]

all_results = Dict{Float64, Float64}()

for p in p_range
    logprintln(logf, "\n=== p = $p ===")
    @time setup = decoder_setup(l, m, A_terms, B_terms, num_cycles, p; osd_order=osd_order)

    logprintln(logf, "  Running $num_total_samples MC trials on $(nthreads()) threads...")
    failures = Atomic{Int}(0)
    @time @threads for i in 1:num_total_samples
        rng = rngs[threadid()]
        run_trial(p, setup, rng) || atomic_add!(failures, 1)
    end

    total_failures = failures[]
    PL = total_failures / num_total_samples
    pL = 1 - (1 - PL)^(1/num_cycles)
    all_results[p] = pL
    logprintln(logf, "  failures=$total_failures/$num_total_samples, P_L=$(round(PL,sigdigits=4)), p_L≈$(round(pL,sigdigits=4))")
end

close(logf)
println("\nLog saved to $logfile")

[[72,12,6]] circuit-level, num_cycles=6, osd_order=1, samples=5088, threads=96

=== p = 0.001 ===
  Gates/cycle: 720
  Enumerating Z faults...
    HdecZ: (288, 2233)
  Enumerating X faults...
    HdecX: (288, 2269)
    After filtering: HdecZ (288, 2232), HdecX (288, 2268)


## Save

In [ ]:
fn = "bb_$(2*l*m)_$(k)_$(d)_circuit_level_osd$(osd_order)_$(num_total_samples).jld2"
jldsave(fn;
    l=l, m=m, A_terms=A_terms, B_terms=B_terms,
    p_range=p_range, num_cycles=num_cycles,
    num_samples=num_total_samples,
    all_results=all_results,
)
println("\nSaved to $fn")

## Plot

In [ ]:
ps = sort(collect(keys(all_results)))
pLs = [all_results[p] for p in ps]
yerr = [sqrt(pL * (1-pL) / num_total_samples) for pL in pLs]

g = plot(xscale=:log10, yscale=:log10, legend=:topleft)
plot!(ps, pLs; label="[[$(2*l*m),$(k),$(d)]] BP+OSD-$(osd_order) circuit-level", marker=:circle)#, yerr=yerr)
# plot!(ps, k .* ps; label="p_L = $(k)p (break-even)", ls=:dash, lw=2, color=:black)
plot!(xlabel="physical error rate p", ylabel="logical error rate p_L")
savefig(g, "bb_$(2*l*m)_$(k)_$(d)_circuit_level_osd$(osd_order)_$(num_total_samples).pdf")
println("Plot saved.")
g

## Collect all runs and plot together

In [ ]:
files = filter(f -> occursin(r"bb.*circuit_level.*\.jld2$", f), readdir())
println("Found $(length(files)) files: ", files)

g = plot(xscale=:log10, yscale=:log10, legend=:topleft,
         xlabel="physical error rate p", ylabel="logical error rate p_L")

for f in sort(files)
    d = load(f)
    res = d["all_results"]
    ps = sort(collect(keys(res)))
    pLs = [res[p] for p in ps]
    # Extract code params from filename
    label = replace(splitext(f)[1], "_" => " ")
    plot!(ps, pLs; label=label, marker=:circle)
end

savefig(g, "bb_circuit_level_all_runs.pdf")
println("Saved bb_circuit_level_all_runs.pdf")
g